In [0]:
%pip install \
    azure-keyvault-secrets==4.7.0 \
    azure-identity==1.15.0 \
    azure-core==1.29.5 \
    azure-storage-file-datalake==12.14.0 \
    sseclient-py \
    openai \
    dotenv \
    confluent-kafka \
    great-expectations \
    altair==4.2.2 \
    redis \
    psycopg2-binary

In [0]:
import os
import sys
import json
from datetime import datetime
from pyspark.sql import SparkSession

os.environ["KEY_VAULT_URL"] = "https://kv-sense-team4.vault.azure.net/"
sys.path.insert(0, "/Workspace/Repos/3dt030@msacademy.msai.kr/3dt-3nd-project/src")

import utils.vault_manager
utils.vault_manager._instance = None
from utils.vault_manager import get_vault_manager

vault = get_vault_manager()
# vault 연결 후 ADLS OAuth 설정 (Databricks 전용)
vault.get_storage_client("datacopsadls")  # Spark conf에 OAuth 설정

# 설정 확인
spark = SparkSession.getActiveSession()

key = "fs.azure.account.auth.type.datacopsadls.dfs.core.windows.net"
print(f"[INFO] ADLS 인증 방식: {spark.conf.get(key, 'NOT SET')}")
# "OAuth" 가 나와야 정상

In [0]:
import psycopg2
import psycopg2.extras
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

ACCOUNT         = "datacopsadls"
BASE_QUARANTINE = f"abfss://quarantine@{ACCOUNT}.dfs.core.windows.net"
BASE_SILVER     = f"abfss://silver@{ACCOUNT}.dfs.core.windows.net"
BASE_MASTER     = f"abfss://master@{ACCOUNT}.dfs.core.windows.net"

def get_pg_conn():
    return psycopg2.connect(
        host=vault.get_secret("db-host"),
        dbname=vault.get_secret("db-name"),
        user=vault.get_secret("db-user"),
        password=vault.get_secret("db-password")
    )

print("[OK] 경로 상수 + PostgreSQL 헬퍼 정의 완료")
print(f"  quarantine : {BASE_QUARANTINE}")
print(f"  silver     : {BASE_SILVER}")
print(f"  master     : {BASE_MASTER}")

In [0]:
DDL = """
CREATE TABLE IF NOT EXISTS quarantine_actions (
    action_id   SERIAL PRIMARY KEY,
    domain      TEXT        NOT NULL,
    file_path   TEXT,
    row_ids     JSONB,
    action      TEXT        NOT NULL DEFAULT 'pending',
    reason      TEXT,
    reviewed_by TEXT,
    reviewed_at TIMESTAMP   DEFAULT now()
);
"""

try:
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute(DDL)
    conn.commit()
    conn.close()
    print("[OK] quarantine_actions 테이블 준비 완료")
except Exception as e:
    print(f"[ERROR] 테이블 생성 실패: {e}")

In [0]:
def list_quarantine_domains() -> list:
    """quarantine 컨테이너에서 도메인 목록 반환"""
    try:
        return [
            f.path.rstrip("/").split("/")[-1]
            for f in dbutils.fs.ls(BASE_QUARANTINE)
            if f.isDir() and not f.name.startswith("_")
        ]
    except Exception as e:
        print(f"[WARN] 도메인 목록 조회 실패: {e}")
        return []


def load_quarantine_df(domain: str):
    """quarantine Delta 테이블 로드. Returns: (DataFrame, count)"""
    path = f"{BASE_QUARANTINE}/{domain}"
    try:
        df = spark.read.parquet(path)
        cnt = df.count()
        print(f"[INFO] {domain} 격리 데이터: {cnt}행")
        return df, cnt
    except Exception as e:
        print(f"[WARN] {domain} 격리 데이터 없음: {e}")
        return None, 0


def get_quarantine_summary(domain: str) -> dict:
    """이유별 집계 + 샘플 5행 반환"""
    df, cnt = load_quarantine_df(domain)
    if df is None or cnt == 0:
        return {"domain": domain, "total": 0, "by_reason": [], "sample": []}

    by_reason = (
        df.groupBy("_quarantine_reason")
          .count()
          .orderBy(F.col("count").desc())
          .limit(20)
          .collect()
    )

    sample_cols = [c for c in df.columns if not c.startswith("_")][:10]
    sample_cols += ["_quarantine_reason"]
    sample = df.select(sample_cols).limit(5).toPandas().to_dict(orient="records")

    return {
        "domain":    domain,
        "total":     cnt,
        "by_reason": [{"reason": r["_quarantine_reason"], "count": r["count"]}
                      for r in by_reason],
        "sample":    sample
    }


# 동작 확인
domains = list_quarantine_domains()
print(f"[INFO] 격리 도메인 목록: {domains}")

In [0]:
def _log_action(domain: str, row_ids, action: str):
    """quarantine_actions 테이블에 액션 기록"""
    try:
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            INSERT INTO quarantine_actions (domain, row_ids, action, reviewed_at)
            VALUES (%s, %s, %s, now())
        """, (domain, json.dumps(row_ids) if row_ids else None, action))
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"[WARN] 액션 로그 기록 실패: {e}")


def delete_quarantine_data(domain: str, row_ids: list = None):
    path = f"{BASE_QUARANTINE}/{domain}"
    try:
        if row_ids:
            # parquet은 행 단위 삭제 불가 → 읽어서 필터 후 덮어쓰기
            df = spark.read.parquet(path)
            df_remain = df.filter(~F.col("_row_hash").isin(row_ids))
            df_remain.write.mode("overwrite").parquet(path)
            print(f"[OK] {domain} | {len(row_ids)}행 삭제 완료")
        else:
            # 전체 삭제는 폴더 삭제
            dbutils.fs.rm(path, recurse=True)
            print(f"[OK] {domain} | 전체 격리 데이터 삭제 완료")
        _log_action(domain, row_ids, "delete")
    except Exception as e:
        print(f"[ERROR] 삭제 실패: {e}")
        raise


def approve_to_master(domain: str, row_ids: list = None):
    q_path      = f"{BASE_QUARANTINE}/{domain}"
    silver_path = f"{BASE_SILVER}/{domain}"
    master_path = f"{BASE_MASTER}/{domain}"

    META_COLS = ["_quarantine_reason", "_ingest_ts", "_source_type",
                 "_platform", "_company", "_domain", "_row_hash",
                 "_quarantine_ts", "_processed_at", "_run_id", "_epoch_id"]

    try:
        # 1) 격리 데이터 로드 (parquet)
        df_q = spark.read.parquet(q_path)
        if row_ids:
            df_q = df_q.filter(F.col("_row_hash").isin(row_ids))

        drop_q      = [c for c in META_COLS if c in df_q.columns]
        df_approved = df_q.drop(*drop_q)

        # 2) Silver 데이터 로드 (parquet)
        try:
            df_silver = spark.read.parquet(silver_path)
            drop_s    = [c for c in META_COLS if c in df_silver.columns]
            df_silver = df_silver.drop(*drop_s)
        except Exception:
            df_silver = None
            print(f"[WARN] Silver 없음 — 격리 승인 데이터만 master에 저장")

        # 3) Silver + 승인 데이터 union
        df_master = df_approved if df_silver is None else df_silver.unionByName(
            df_approved, allowMissingColumns=True
        )
        df_master = df_master.withColumn("_merged_at", F.current_timestamp())

        # 4) master에 parquet append
        df_master.write \
            .mode("append") \
            .parquet(master_path)

        cnt = df_approved.count()
        print(f"[OK] {domain} | {cnt}행 master 병합 완료 → {master_path}")
        _log_action(domain, row_ids, "approve")

        # 5) 승인된 행 quarantine에서 제거
        delete_quarantine_data(domain, row_ids)

    except Exception as e:
        print(f"[ERROR] master 병합 실패: {e}")
        raise


print("[OK] 액션 처리 함수 정의 완료")

In [0]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import uvicorn, threading

app_review = FastAPI(title="DataCops Quarantine Review API")

class ActionRequest(BaseModel):
    row_ids: Optional[List[str]] = None  # None이면 전체 대상


@app_review.get("/api/quarantine/domains")
def api_list_domains():
    return {"domains": list_quarantine_domains()}


@app_review.get("/api/quarantine/{domain}")
def api_get_quarantine(domain: str):
    summary = get_quarantine_summary(domain)
    if summary["total"] == 0:
        raise HTTPException(status_code=404, detail=f"{domain} 격리 데이터 없음")
    return summary


@app_review.post("/api/quarantine/{domain}/approve")
def api_approve(domain: str, req: ActionRequest):
    try:
        approve_to_master(domain, req.row_ids)
        target = f"{len(req.row_ids)}행" if req.row_ids else "전체"
        return {"status": "ok", "message": f"{domain} | {target} master 병합 완료"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app_review.post("/api/quarantine/{domain}/delete")
def api_delete(domain: str, req: ActionRequest):
    try:
        delete_quarantine_data(domain, req.row_ids)
        target = f"{len(req.row_ids)}행" if req.row_ids else "전체"
        return {"status": "ok", "message": f"{domain} | {target} 삭제 완료"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app_review.get("/api/quarantine/{domain}/actions")
def api_get_actions(domain: str):
    try:
        conn = get_pg_conn()
        cur  = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
        cur.execute("""
            SELECT action_id, domain, row_ids, action, reviewed_by, reviewed_at
            FROM quarantine_actions
            WHERE domain = %s
            ORDER BY reviewed_at DESC
            LIMIT 50
        """, (domain,))
        rows = cur.fetchall()
        conn.close()
        return {"domain": domain, "actions": [dict(r) for r in rows]}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("[OK] FastAPI 엔드포인트 정의 완료")
print("  GET  /api/quarantine/domains")
print("  GET  /api/quarantine/{domain}")
print("  POST /api/quarantine/{domain}/approve")
print("  POST /api/quarantine/{domain}/delete")
print("  GET  /api/quarantine/{domain}/actions")

In [0]:
PORT = 8000

def _run_server():
    uvicorn.run(app_review, host="0.0.0.0", port=PORT, log_level="warning")

t = threading.Thread(target=_run_server, daemon=True)
t.start()

import time; time.sleep(2)
print(f"[OK] Quarantine Review API 실행 중 (port={PORT})")
print(f"  Swagger UI → http://0.0.0.0:{PORT}/docs")

In [0]:
# 도메인 목록 확인
domains = list_quarantine_domains()
print(f"격리 도메인: {domains}")

if domains:
    test_domain = domains[0]

    # 요약 확인
    summary = get_quarantine_summary(test_domain)
    print(f"\n[{test_domain}] 격리 요약")
    print(f"  총 건수: {summary['total']}")
    for r in summary["by_reason"]:
        print(f"  - {r['reason']}: {r['count']}건")

    # 전체 승인 테스트 (주의: 실제 실행됨, 주석 풀고 사용)
    approve_to_master(test_domain)

    # 전체 삭제 테스트 (주의: 실제 실행됨, 주석 풀고 사용)
    # delete_quarantine_data(test_domain)